# Lab 4 — ONNX inference and PyTorch parity
Use your Lab 3 `gesture_model.pth` and `gesture_model.onnx` and compare identical **real** held-out feature samples. Benchmark same device, same inputs.


In [ ]:
!pip -q install torch onnx onnxruntime numpy pandas


In [ ]:
from google.colab import files
uploaded=files.upload() # select gesture_model.onnx; optionally select a session CSV
import os, numpy as np, onnx, onnxruntime as ort, time, pandas as pd
model_path='gesture_model.onnx'
assert os.path.exists(model_path), 'Please upload gesture_model.onnx'
onnx.checker.check_model(onnx.load(model_path))
options=ort.SessionOptions();options.intra_op_num_threads=1
sess=ort.InferenceSession(model_path,options,providers=['CPUExecutionProvider'])
features=np.zeros((1,42),dtype=np.float32)
print('Input',sess.get_inputs()[0].name,sess.get_inputs()[0].shape)
print('Logits',sess.run(None,{sess.get_inputs()[0].name:features})[0])


In [ ]:
uploaded=files.upload() # upload gesture_model.pth and one CSV for parity
import torch, torch.nn as nn
class GestureMLP(nn.Module):
  def __init__(self,dims):
    super().__init__();blocks=[]
    for i in range(len(dims)-1):
      blocks.append(nn.Linear(dims[i],dims[i+1]))
      if i<len(dims)-2:blocks.append(nn.ReLU())
    self.layers=nn.Sequential(*blocks)
  def forward(self,x):return self.layers(x)
ckpt=torch.load('gesture_model.pth',map_location='cpu',weights_only=False)
# Layer sizes come from the checkpoint, so Tiny, Small, or Large from Lab 3 all load.
weights=[v for k,v in ckpt['state_dict'].items() if k.endswith('weight')]
dims=[weights[0].shape[1]]+[w.shape[0] for w in weights]
model=GestureMLP(dims);model.load_state_dict(ckpt['state_dict']);model.eval()
print('Architecture:','→'.join(map(str,dims)))
csvs=[n for n in uploaded if n.endswith('.csv')];assert csvs,'Upload a feature CSV'
df=pd.read_csv(csvs[0]);x=df[[f'f{i}' for i in range(42)]].to_numpy(dtype=np.float32)[:32]
with torch.no_grad():pytorch_out=model(torch.from_numpy(x)).numpy()
onnx_out=sess.run(None,{sess.get_inputs()[0].name:x})[0]
print('Max absolute difference:',np.max(np.abs(pytorch_out-onnx_out)))
print('Same predicted class:',np.mean(pytorch_out.argmax(1)==onnx_out.argmax(1)))
assert np.allclose(pytorch_out,onnx_out,rtol=1e-4,atol=1e-5)


In [ ]:
# Independent timing; compare batch=1 and batch=32, report median (not best-case).
def benchmark(fn,n=120):
  for _ in range(15):fn()
  readings=[]
  for _ in range(n):
    t=time.perf_counter();fn();readings.append((time.perf_counter()-t)*1000)
  return np.median(readings),np.percentile(readings,95)
for batch in [1,32]:
  arr=x[:batch] if len(x)>=batch else np.repeat(x[:1],batch,axis=0)
  med,p95=benchmark(lambda:sess.run(None,{sess.get_inputs()[0].name:arr}))
  print(f'ONNX CPU batch={batch}: median={med:.4f} ms, p95={p95:.4f} ms')
  with torch.no_grad():
    med2,p952=benchmark(lambda:model(torch.from_numpy(arr)))
  print(f'PyTorch CPU batch={batch}: median={med2:.4f} ms, p95={p952:.4f} ms')
# TODO: explain why ONNX may or may not be faster for a tiny MLP.
